# Diversity Pipeline Analysis

## Objective
Analyze recruiting funnel for potential bias and identify where diverse candidates may be disproportionately filtered out:
- Track demographic composition at each pipeline stage
- Compare application → hire conversion rates by group
- Identify stages with disproportionate attrition
- Analyze interview scores for bias patterns

## Key Questions
1. Where do we lose diverse candidates in the funnel?
2. Are conversion rates equitable across demographics?
3. Do interview scores vary systematically by group?
4. Which sources produce diverse candidate pools?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load Data

In [ ]:
# Load datasets
candidates_df = pd.read_csv('../data/candidates.csv')
interviews_df = pd.read_csv('../data/interviews.csv')
hires_df = pd.read_csv('../data/hires.csv')

print(f"📊 Data loaded:")
print(f"   Candidates: {len(candidates_df):,}")
print(f"   Interviews: {len(interviews_df):,}")
print(f"   Hires: {len(hires_df):,}")

# Overview of demographics
print("\n👥 Candidate Pool Demographics:\n")
print("Gender distribution:")
print(candidates_df['gender'].value_counts())
print("\nEthnicity distribution:")
print(candidates_df['ethnicity'].value_counts())

candidates_df.head()

## 2. Overall Diversity Metrics

In [ ]:
# Calculate representation at each stage
def get_diversity_metrics(df, stage_name):
    gender_dist = df['gender'].value_counts(normalize=True) * 100
    ethnicity_dist = df['ethnicity'].value_counts(normalize=True) * 100
    
    return {
        'stage': stage_name,
        'total': len(df),
        'female_%': gender_dist.get('Female', 0),
        'male_%': gender_dist.get('Male', 0),
        'underrepresented_ethnicity_%': (
            ethnicity_dist.get('Hispanic/Latino', 0) + 
            ethnicity_dist.get('Black/African American', 0) +
            ethnicity_dist.get('Native American', 0) +
            ethnicity_dist.get('Two or more races', 0)
        )
    }

# Calculate for each stage
pipeline_diversity = []

# Applications
pipeline_diversity.append(get_diversity_metrics(candidates_df, 'Applications'))

# Phone Screen
phone_screen = candidates_df[candidates_df['current_stage'].isin(['Phone Screen', 'Technical Interview', 
                                                                   'Onsite Interview', 'Offer', 'Hired'])]
pipeline_diversity.append(get_diversity_metrics(phone_screen, 'Phone Screen+'))

# Onsite
onsite = candidates_df[candidates_df['current_stage'].isin(['Onsite Interview', 'Offer', 'Hired'])]
pipeline_diversity.append(get_diversity_metrics(onsite, 'Onsite+'))

# Offers
offers = candidates_df[candidates_df['current_stage'].isin(['Offer', 'Hired'])]
pipeline_diversity.append(get_diversity_metrics(offers, 'Offer'))

# Hires
hired = candidates_df[candidates_df['current_stage'] == 'Hired']
pipeline_diversity.append(get_diversity_metrics(hired, 'Hired'))

diversity_df = pd.DataFrame(pipeline_diversity).round(1)

print("📊 Diversity Across Pipeline Stages:\n")
print(diversity_df.to_string(index=False))

In [ ]:
# Visualize diversity flow
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gender representation
stages = diversity_df['stage']
x = range(len(stages))

ax1.plot(x, diversity_df['female_%'], marker='o', linewidth=3, markersize=10, 
         label='Female', color='purple')
ax1.plot(x, diversity_df['male_%'], marker='s', linewidth=3, markersize=10, 
         label='Male', color='steelblue')
ax1.set_xticks(x)
ax1.set_xticklabels(stages, rotation=45, ha='right')
ax1.set_ylabel('Representation (%)')
ax1.set_title('Gender Representation Across Pipeline', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 100)

# Underrepresented ethnicity
ax2.plot(x, diversity_df['underrepresented_ethnicity_%'], marker='o', linewidth=3, 
         markersize=10, color='green', label='Underrepresented Groups')
ax2.axhline(y=diversity_df['underrepresented_ethnicity_%'].iloc[0], 
            color='red', linestyle='--', linewidth=2, alpha=0.5, 
            label='Application Baseline')
ax2.set_xticks(x)
ax2.set_xticklabels(stages, rotation=45, ha='right')
ax2.set_ylabel('Representation (%)')
ax2.set_title('Underrepresented Ethnicity Across Pipeline', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Add percentage change annotations
for i in range(len(diversity_df)-1):
    change = diversity_df['underrepresented_ethnicity_%'].iloc[i+1] - diversity_df['underrepresented_ethnicity_%'].iloc[i]
    if abs(change) > 1:
        color = 'red' if change < 0 else 'green'
        ax2.annotate(f'{change:+.1f}%', 
                    xy=(i+0.5, diversity_df['underrepresented_ethnicity_%'].iloc[i]),
                    fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Conversion Rate Analysis by Gender

In [ ]:
# Calculate hire rates by gender
gender_conversion = candidates_df.groupby('gender').agg({
    'candidate_id': 'count',
    'current_stage': lambda x: (x == 'Hired').sum()
}).rename(columns={'candidate_id': 'applications', 'current_stage': 'hires'})

gender_conversion['hire_rate_%'] = (gender_conversion['hires'] / 
                                     gender_conversion['applications'] * 100).round(2)

print("📊 Hire Rate by Gender:\n")
print(gender_conversion)

# Statistical test: Chi-square test for independence
# Focus on Female vs Male for clarity
female_data = gender_conversion.loc['Female']
male_data = gender_conversion.loc['Male']

contingency_table = np.array([
    [female_data['hires'], female_data['applications'] - female_data['hires']],
    [male_data['hires'], male_data['applications'] - male_data['hires']]
])

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\n📈 Statistical Test (Female vs Male):")
print(f"   Chi-square: {chi2:.3f}")
print(f"   p-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"   ⚠️ Statistically significant difference in hire rates")
    diff = female_data['hire_rate_%'] - male_data['hire_rate_%']
    if diff > 0:
        print(f"   Female hire rate is {diff:.2f} percentage points HIGHER")
    else:
        print(f"   Female hire rate is {abs(diff):.2f} percentage points LOWER")
else:
    print(f"   ✅ No statistically significant difference in hire rates")

In [ ]:
# Visualize conversion rates
fig, ax = plt.subplots(figsize=(12, 6))

# Exclude 'Prefer not to say' for clarity
plot_data = gender_conversion[gender_conversion.index != 'Prefer not to say']

x = range(len(plot_data))
bars = ax.bar(x, plot_data['hire_rate_%'], color=['purple', 'steelblue', 'orange'][:len(plot_data)],
              alpha=0.7, edgecolor='black', linewidth=2)

# Add overall average line
overall_rate = (candidates_df['current_stage'] == 'Hired').sum() / len(candidates_df) * 100
ax.axhline(y=overall_rate, color='red', linestyle='--', linewidth=2, 
           label=f'Overall Avg: {overall_rate:.2f}%')

ax.set_xticks(x)
ax.set_xticklabels(plot_data.index, fontsize=12)
ax.set_ylabel('Hire Rate (%)', fontsize=12)
ax.set_title('Hire Rate by Gender', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels and sample size
for i, (rate, n) in enumerate(zip(plot_data['hire_rate_%'], plot_data['applications'])):
    ax.text(i, rate + 0.2, f'{rate:.2f}%', ha='center', fontsize=11, fontweight='bold')
    ax.text(i, -0.5, f'n={n}', ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

## 4. Conversion Rate Analysis by Ethnicity

In [ ]:
# Calculate hire rates by ethnicity
ethnicity_conversion = candidates_df.groupby('ethnicity').agg({
    'candidate_id': 'count',
    'current_stage': lambda x: (x == 'Hired').sum()
}).rename(columns={'candidate_id': 'applications', 'current_stage': 'hires'})

ethnicity_conversion['hire_rate_%'] = (ethnicity_conversion['hires'] / 
                                        ethnicity_conversion['applications'] * 100).round(2)
ethnicity_conversion = ethnicity_conversion.sort_values('hire_rate_%', ascending=False)

print("📊 Hire Rate by Ethnicity:\n")
print(ethnicity_conversion)

# Calculate variance from overall average
ethnicity_conversion['diff_from_avg'] = (ethnicity_conversion['hire_rate_%'] - overall_rate).round(2)

print(f"\n📈 Deviation from Overall Average ({overall_rate:.2f}%):")
for ethnicity, diff in ethnicity_conversion['diff_from_avg'].items():
    if ethnicity != 'Prefer not to say':
        symbol = "↑" if diff > 0 else "↓" if diff < 0 else "="
        print(f"   {ethnicity}: {symbol} {diff:+.2f} percentage points")

In [ ]:
# Visualize ethnicity conversion rates
fig, ax = plt.subplots(figsize=(14, 7))

# Exclude 'Prefer not to say'
plot_data = ethnicity_conversion[ethnicity_conversion.index != 'Prefer not to say']

# Color code by deviation from average
colors = ['green' if x > 0 else 'red' if x < -1 else 'gray' 
          for x in plot_data['diff_from_avg']]

y_pos = range(len(plot_data))
bars = ax.barh(y_pos, plot_data['hire_rate_%'], color=colors, alpha=0.7, edgecolor='black')

# Add overall average line
ax.axvline(x=overall_rate, color='blue', linestyle='--', linewidth=2, 
           label=f'Overall Avg: {overall_rate:.2f}%', zorder=5)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data.index, fontsize=11)
ax.set_xlabel('Hire Rate (%)', fontsize=12)
ax.set_title('Hire Rate by Ethnicity', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (rate, n, diff) in enumerate(zip(plot_data['hire_rate_%'], 
                                        plot_data['applications'],
                                        plot_data['diff_from_avg'])):
    ax.text(rate + 0.2, i, f'{rate:.2f}% (n={n})', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🟢 Green = Above average | 🔴 Red = Below average | ⚫ Gray = Near average")

## 5. Interview Score Analysis by Demographics

In [ ]:
# Merge interview data with candidate demographics
interviews_with_demo = interviews_df.merge(
    candidates_df[['candidate_id', 'gender', 'ethnicity']], 
    on='candidate_id'
)

# Average interview score by gender
gender_scores = interviews_with_demo.groupby('gender')['score'].agg(['mean', 'std', 'count']).round(2)

print("📊 Average Interview Score by Gender:\n")
print(gender_scores)

# Statistical test: ANOVA for gender
female_scores = interviews_with_demo[interviews_with_demo['gender'] == 'Female']['score']
male_scores = interviews_with_demo[interviews_with_demo['gender'] == 'Male']['score']

t_stat, p_value_gender = stats.ttest_ind(female_scores, male_scores)

print(f"\n📈 T-test (Female vs Male interview scores):")
print(f"   t-statistic: {t_stat:.3f}")
print(f"   p-value: {p_value_gender:.4f}")
if p_value_gender < 0.05:
    print(f"   ⚠️ Statistically significant difference in interview scores")
    diff = gender_scores.loc['Female', 'mean'] - gender_scores.loc['Male', 'mean']
    print(f"   Female scores are {diff:+.2f} points different")
else:
    print(f"   ✅ No statistically significant difference in interview scores")

In [ ]:
# Average interview score by ethnicity
ethnicity_scores = interviews_with_demo.groupby('ethnicity')['score'].agg(['mean', 'std', 'count']).round(2)
ethnicity_scores = ethnicity_scores.sort_values('mean', ascending=False)

print("📊 Average Interview Score by Ethnicity:\n")
print(ethnicity_scores)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gender scores
gender_plot = gender_scores[gender_scores.index.isin(['Female', 'Male', 'Non-binary'])]
ax1.bar(range(len(gender_plot)), gender_plot['mean'], 
           color=['purple', 'steelblue', 'orange'][:len(gender_plot)],
           alpha=0.7, edgecolor='black', linewidth=2)
ax1.axhline(y=interviews_df['score'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Overall Avg: {interviews_df["score"].mean():.2f}')
ax1.set_xticks(range(len(gender_plot)))
ax1.set_xticklabels(gender_plot.index)
ax1.set_ylabel('Average Interview Score')
ax1.set_title('Interview Score by Gender', fontsize=14, fontweight='bold')
ax1.set_ylim(1, 5)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, (score, n) in enumerate(zip(gender_plot['mean'], gender_plot['count'])):
    ax1.text(i, score + 0.05, f'{score:.2f}', ha='center', fontsize=11, fontweight='bold')
    ax1.text(i, 1.1, f'n={int(n)}', ha='center', fontsize=9, color='gray')

# Ethnicity scores
ethnicity_plot = ethnicity_scores[ethnicity_scores.index != 'Prefer not to say']
y_pos = range(len(ethnicity_plot))
ax2.barh(y_pos, ethnicity_plot['mean'], color='green', alpha=0.7, edgecolor='black')
ax2.axvline(x=interviews_df['score'].mean(), color='red', linestyle='--', 
           linewidth=2, label=f'Overall Avg: {interviews_df["score"].mean():.2f}')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(ethnicity_plot.index, fontsize=10)
ax2.set_xlabel('Average Interview Score')
ax2.set_title('Interview Score by Ethnicity', fontsize=14, fontweight='bold')
ax2.set_xlim(1, 5)
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, (score, n) in enumerate(zip(ethnicity_plot['mean'], ethnicity_plot['count'])):
    ax2.text(score + 0.05, i, f'{score:.2f} (n={int(n)})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Source-of-Hire Diversity Analysis

In [ ]:
# Which sources produce diverse candidate pools?
source_diversity = []

for source in candidates_df['source'].unique():
    source_data = candidates_df[candidates_df['source'] == source]
    
    female_pct = (source_data['gender'] == 'Female').sum() / len(source_data) * 100
    
    underrep_ethnicity = (
        (source_data['ethnicity'] == 'Hispanic/Latino').sum() +
        (source_data['ethnicity'] == 'Black/African American').sum() +
        (source_data['ethnicity'] == 'Native American').sum() +
        (source_data['ethnicity'] == 'Two or more races').sum()
    ) / len(source_data) * 100
    
    source_diversity.append({
        'source': source,
        'total_candidates': len(source_data),
        'female_%': female_pct,
        'underrep_ethnicity_%': underrep_ethnicity
    })

source_div_df = pd.DataFrame(source_diversity).sort_values('female_%', ascending=False).round(1)

print("📊 Source Diversity Metrics:\n")
print(source_div_df.to_string(index=False))

# Identify most/least diverse sources
most_diverse_gender = source_div_df.iloc[0]['source']
least_diverse_gender = source_div_df.iloc[-1]['source']

most_diverse_ethnicity = source_div_df.sort_values('underrep_ethnicity_%', ascending=False).iloc[0]['source']

print(f"\n💡 Key Findings:")
print(f"   Most gender-diverse source: {most_diverse_gender} ({source_div_df[source_div_df['source']==most_diverse_gender]['female_%'].values[0]:.1f}% female)")
print(f"   Least gender-diverse source: {least_diverse_gender} ({source_div_df[source_div_df['source']==least_diverse_gender]['female_%'].values[0]:.1f}% female)")
print(f"   Most ethnically diverse source: {most_diverse_ethnicity} ({source_div_df[source_div_df['source']==most_diverse_ethnicity]['underrep_ethnicity_%'].values[0]:.1f}% underrepresented)")

In [ ]:
# Visualize source diversity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Gender diversity by source
overall_female_pct = (candidates_df['gender'] == 'Female').sum() / len(candidates_df) * 100

y_pos = range(len(source_div_df))
colors1 = ['green' if x > overall_female_pct else 'orange' for x in source_div_df['female_%']]
ax1.barh(y_pos, source_div_df['female_%'], color=colors1, alpha=0.7, edgecolor='black')
ax1.axvline(x=overall_female_pct, color='red', linestyle='--', linewidth=2,
           label=f'Overall: {overall_female_pct:.1f}%')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(source_div_df['source'], fontsize=10)
ax1.set_xlabel('Female Representation (%)')
ax1.set_title('Gender Diversity by Source', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Ethnicity diversity by source
overall_underrep_pct = (
    (candidates_df['ethnicity'] == 'Hispanic/Latino').sum() +
    (candidates_df['ethnicity'] == 'Black/African American').sum() +
    (candidates_df['ethnicity'] == 'Native American').sum() +
    (candidates_df['ethnicity'] == 'Two or more races').sum()
) / len(candidates_df) * 100

source_div_df_eth = source_div_df.sort_values('underrep_ethnicity_%', ascending=True)
y_pos2 = range(len(source_div_df_eth))
colors2 = ['green' if x > overall_underrep_pct else 'orange' 
           for x in source_div_df_eth['underrep_ethnicity_%']]
ax2.barh(y_pos2, source_div_df_eth['underrep_ethnicity_%'], color=colors2, 
        alpha=0.7, edgecolor='black')
ax2.axvline(x=overall_underrep_pct, color='red', linestyle='--', linewidth=2,
           label=f'Overall: {overall_underrep_pct:.1f}%')
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(source_div_df_eth['source'], fontsize=10)
ax2.set_xlabel('Underrepresented Ethnicity (%)')
ax2.set_title('Ethnic Diversity by Source', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🟢 Green = Above average diversity | 🟠 Orange = Below average diversity")

## 7. Key Findings & Recommendations

In [ ]:
print("="*70)
print("🎯 KEY FINDINGS")
print("="*70)

print("\n1️⃣ PIPELINE REPRESENTATION:\n")
app_female = diversity_df[diversity_df['stage'] == 'Applications']['female_%'].values[0]
hired_female = diversity_df[diversity_df['stage'] == 'Hired']['female_%'].values[0]
female_change = hired_female - app_female

print(f"   Gender (Female):")
print(f"      Applications: {app_female:.1f}%")
print(f"      Hired: {hired_female:.1f}%")
print(f"      Change: {female_change:+.1f} percentage points")
if female_change < -5:
    print(f"      ⚠️ Significant drop - potential pipeline leakage")
elif female_change > 5:
    print(f"      ✅ Improvement through pipeline")
else:
    print(f"      ✅ Maintained representation")

app_underrep = diversity_df[diversity_df['stage'] == 'Applications']['underrepresented_ethnicity_%'].values[0]
hired_underrep = diversity_df[diversity_df['stage'] == 'Hired']['underrepresented_ethnicity_%'].values[0]
underrep_change = hired_underrep - app_underrep

print(f"\n   Ethnicity (Underrepresented):")
print(f"      Applications: {app_underrep:.1f}%")
print(f"      Hired: {hired_underrep:.1f}%")
print(f"      Change: {underrep_change:+.1f} percentage points")
if underrep_change < -5:
    print(f"      ⚠️ Significant drop - potential pipeline leakage")
elif underrep_change > 5:
    print(f"      ✅ Improvement through pipeline")
else:
    print(f"      ✅ Maintained representation")

print("\n2️⃣ CONVERSION RATE EQUITY:\n")
female_hire_rate = gender_conversion.loc['Female', 'hire_rate_%']
male_hire_rate = gender_conversion.loc['Male', 'hire_rate_%']
hire_rate_diff = female_hire_rate - male_hire_rate

print(f"   Female hire rate: {female_hire_rate:.2f}%")
print(f"   Male hire rate: {male_hire_rate:.2f}%")
print(f"   Difference: {hire_rate_diff:+.2f} percentage points")
if abs(hire_rate_diff) > 1 and p_value < 0.05:
    print(f"   ⚠️ Statistically significant difference (p={p_value:.4f})")
else:
    print(f"   ✅ No significant difference (p={p_value:.4f})")

print("\n3️⃣ INTERVIEW SCORING:\n")
female_int_score = gender_scores.loc['Female', 'mean']
male_int_score = gender_scores.loc['Male', 'mean']
score_diff = female_int_score - male_int_score

print(f"   Female avg score: {female_int_score:.2f}")
print(f"   Male avg score: {male_int_score:.2f}")
print(f"   Difference: {score_diff:+.2f} points")
if abs(score_diff) > 0.2 and p_value_gender < 0.05:
    print(f"   ⚠️ Statistically significant difference (p={p_value_gender:.4f})")
    print(f"   → Review interview scoring for potential bias")
else:
    print(f"   ✅ No significant difference (p={p_value_gender:.4f})")

print("\n4️⃣ SOURCE DIVERSITY:\n")
print(f"   Most gender-diverse: {most_diverse_gender}")
print(f"   Most ethnically diverse: {most_diverse_ethnicity}")
print(f"   Least gender-diverse: {least_diverse_gender}")

print("\n" + "="*70)
print("💡 RECOMMENDATIONS")
print("="*70)

print("\n🎯 IMMEDIATE ACTIONS (0-30 days):\n")
if female_change < -5 or underrep_change < -5:
    print("   1. ⚠️ URGENT: Investigate pipeline leakage")
    print("      → Analyze drop-off points (screening, phone, onsite)")
    print("      → Review rejection reasons by demographic")
else:
    print("   1. ✅ Continue monitoring pipeline representation monthly")

print("   2. Expand recruiting from high-diversity sources")
print(f"      → Increase investment in: {most_diverse_gender}, {most_diverse_ethnicity}")
print("   3. Implement blind resume screening")
print("      → Remove names, schools, photos from initial review")

print("\n🔧 PROCESS IMPROVEMENTS (30-90 days):\n")
print("   1. STRUCTURED INTERVIEWS: Reduce subjective bias")
print("   2. DIVERSE INTERVIEW PANELS: Multiple perspectives")
print("   3. CALIBRATION TRAINING: Unconscious bias workshops")
print("   4. SCORECARD AUDITS: Review for systematic patterns")

print("\n📊 METRICS TO TRACK (MONTHLY):\n")
print("   • Demographic composition at each pipeline stage")
print("   • Hire rate by gender and ethnicity")
print("   • Interview score distributions by demographic")
print("   • Source diversity metrics")
print("   • Offer acceptance rates by group")

print("\n⚖️ COMPLIANCE & BEST PRACTICES:\n")
print("   • Document analysis methods for OFCCP compliance")
print("   • Conduct adverse impact analysis (4/5ths rule)")
print("   • Regular legal review of hiring practices")
print("   • Transparency with candidates about DEI commitment")

print("\n" + "="*70)

## Next Steps

1. **Deep Dive**: Interview candidates who dropped out to understand reasons
2. **Bias Testing**: Conduct resume experiments (identical resumes, different names)
3. **Interviewer Training**: Unconscious bias and structured interviewing workshops
4. **Source Expansion**: Partner with diversity-focused organizations
5. **Continuous Monitoring**: Dashboard tracking diversity metrics in real-time

## Related Analyses
- `01_source_of_hire_analysis.ipynb` - Source effectiveness and diversity
- `04_interview_effectiveness.ipynb` - Interview bias patterns
- `02_time_to_fill_analysis.ipynb` - Process delays by demographic